In [1]:
from airflow.sdk import dag, task
from datetime import datetime, timedelta
from airflow import DAG
from airflow.providers.standard.operators.python import PythonOperator
from airflow.providers.standard.operators.empty import EmptyOperator
from airflow.providers.standard.operators.bash import BashOperator
from airflow.sdk import Param
from airflow.sdk.definitions.param import ParamsDict
from sqlalchemy import create_engine, VARCHAR, Integer,Date, inspect
import os
import pandas as pd
from datetime import datetime
import zipfile
import numpy as np
import requests
from datetime import datetime
from pathlib import Path
import shutil
import boto3
from botocore.config import Config


c:\Users\Nelio\anaconda3\Lib\site-packages\airflow\__init__.py:45: RuntimeWarning: Airflow currently can be run on POSIX-compliant Operating Systems. For development, it is regularly tested on fairly modern Linux Distros and recent versions of macOS. On Windows you can run it via WSL2 (Windows Subsystem for Linux 2) or via Linux Containers. The work to add Windows support is tracked via https://github.com/apache/airflow/issues/10388, but it is not a high priority.
  warnings.warn(
2026-04-15T13:31:31.445308Z [info     ] setup plugin alembic.autogenerate.schemas [alembic.runtime.plugins] loc=plugins.py:37
2026-04-15T13:31:31.452553Z [info     ] setup plugin alembic.autogenerate.tables [alembic.runtime.plugins] loc=plugins.py:37
2026-04-15T13:31:31.460138Z [info     ] setup plugin alembic.autogenerate.types [alembic.runtime.plugins] loc=plugins.py:37
2026-04-15T13:31:31.467672Z [info     ] setup plugin alembic.autogenerate.constraints [alembic.runtime.plugins] loc=plugins.py:37
2026-04-1

In [2]:
def unzip_file(zip_path, file_save):
    os.makedirs(file_save, exist_ok=True)

    for file_name in os.listdir(zip_path):
        if file_name.lower().endswith('.zip'):
            zip_file = os.path.join(zip_path, file_name)
            with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                for member in zip_ref.namelist():
                    nome_arquivo = os.path.basename(member)
                    destino_final = os.path.join(file_save, nome_arquivo)
                    with zip_ref.open(member) as origem, open(destino_final, 'wb') as destino:
                        shutil.copyfileobj(origem, destino)
                    print(f"Extraído: {nome_arquivo}")

In [39]:
def perfil (file_csv):
    # engine = create_engine(f"postgresql+psycopg2://{USER}:{PASSWORD}@db:5432/eleicao_RR")
    colunas_leitura = [
    'dt_geracao', 'hh_geracao', 'ano_eleicao', 'sg_uf', 'cd_municipio',
    'nm_municipio', 'nr_zona', 'nr_secao', 'nr_local_votacao',
    'nm_local_votacao', 'cd_genero', 'ds_genero', 'cd_estado_civil',
    'ds_estado_civil', 'cd_faixa_etaria', 'ds_faixa_etaria',
    'cd_grau_escolaridade', 'ds_grau_escolaridade', 'cd_raca_cor',
    'ds_raca_cor', 'cd_identidade_genero', 'ds_identidade_genero',
    'cd_quilombola', 'ds_quilombola', 'cd_interprete_libras',
    'ds_interprete_libras', 'tp_obrigatoriedade_voto',
    'qt_eleitores_perfil', 'qt_eleitores_biometria',
    'qt_eleitores_deficiencia', 'qt_eleitores_inc_nm_social',
    'id_zona_secao_municipio'
    ]
    dfs = []
    colunas_num = ['qt_eleitores_inc_nm_social', 'qt_eleitores_deficiencia', 'qt_eleitores_biometria', 'qt_eleitores_perfil']
    colunas_data = ['dt_geracao']
    colunas_hora = ['hh_geracao']
    for bweb in os.listdir(file_csv):
        full_path = os.path.join(file_csv, bweb)
        if full_path.endswith('.csv'):
            print(f"Lendo arquivo: {full_path}")
            df = pd.read_csv(full_path, encoding='latin-1', sep=';',dtype=str,usecols=lambda col: col.strip().lower() in colunas_leitura,nrows=10)
            df.columns = [col.strip().lower() for col in df.columns]
            for col in colunas_num:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col]).astype('Int64')
            for col in colunas_data:
                if col in df.columns:
                    df[col] = pd.to_datetime(df[col], dayfirst=True, format='%d/%m/%Y', errors='coerce').dt.date
            for col in colunas_hora:
                if col in df.columns:
                    df[col] = pd.to_datetime(df[col], format='%H:%M:%S', errors='coerce').dt.time
            df['id_zona_secao_municipio'] = df['ano_eleicao'] + df['nr_zona'] + df['nr_secao'] + df['cd_municipio']
            dfs.append(df)
            print(f"Carregando o arquivo no banco de archives: {bweb}")
            # df.to_sql("perfil_eleitorado", engine, index=False, if_exists='append')
            print(f"Arquivo Carregado no Bando de archives: {bweb}")

    return df

In [40]:
perfil(file_csv = r'C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\perfil_eleitorado_secao\csv')

Lendo arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\perfil_eleitorado_secao\csv\perfil_eleitor_secao_2020_RR.csv
Carregando o arquivo no banco de archives: perfil_eleitor_secao_2020_RR.csv
Arquivo Carregado no Bando de archives: perfil_eleitor_secao_2020_RR.csv
Lendo arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\perfil_eleitorado_secao\csv\perfil_eleitor_secao_2022_RR.csv
Carregando o arquivo no banco de archives: perfil_eleitor_secao_2022_RR.csv
Arquivo Carregado no Bando de archives: perfil_eleitor_secao_2022_RR.csv
Lendo arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\perfil_eleitorado_secao\csv\perfil_eleitor_secao_2024_RR.csv
Carregando o arquivo no banco de archives: perfil_eleitor_secao_2024_RR.csv
Arquivo Carregado no Bando de archives: perfil_eleitor_secao_2024_RR.csv


,dt_geracao,hh_geracao,ano_eleicao,sg_uf,cd_municipio,nm_municipio,nr_zona,nr_secao,nr_local_votacao,nm_local_votacao,...,cd_quilombola,ds_quilombola,cd_interprete_libras,ds_interprete_libras,tp_obrigatoriedade_voto,qt_eleitores_perfil,qt_eleitores_biometria,qt_eleitores_deficiencia,qt_eleitores_inc_nm_social,id_zona_secao_municipio
0,2024-09-13,15:41:03,2024,RR,3018,BOA VISTA,1,599,3441,ESCOLA MUNICIPAL PROFESSORA CARMEM EUGENIA MACAGI,...,-1,NÃO INFORMADO,-1,NÃO INFORMADO,Facultativo,1,1,0,0,202415993018
1,2024-09-13,15:41:03,2024,RR,3018,BOA VISTA,5,274,1643,ESCOLA ESTADUAL MARIA SONIA DE BRITO OLIVA,...,-1,NÃO INFORMADO,-1,NÃO INFORMADO,Obrigatório,11,11,0,0,202452743018
2,2024-09-13,15:41:03,2024,RR,3026,IRACEMA,6,10,1031,ESCOLA MUNICIPAL IRACEMA AGUIAR PEREIRA,...,-1,NÃO INFORMADO,-1,NÃO INFORMADO,Obrigatório,1,1,0,0,20246103026
3,2024-09-13,15:41:03,2024,RR,3034,CARACARAÍ,2,123,1228,ESCOLA ESTADUAL JOSÉ VIEIRA DE SALES GUERRA,...,-1,NÃO INFORMADO,-1,NÃO INFORMADO,Obrigatório,3,3,0,0,202421233034
4,2024-09-13,15:41:03,2024,RR,3069,CANTÁ,5,527,1252,ESCOLA ESTADUAL BARBOSA DE ALENCAR,...,-1,NÃO INFORMADO,-1,NÃO INFORMADO,Obrigatório,2,2,0,0,202455273069
5,2024-09-13,15:41:03,2024,RR,3018,BOA VISTA,1,785,3514,ESCOLA ESTADUAL PROFESSOR CARLOS CASADIO,...,-1,NÃO INFORMADO,-1,NÃO INFORMADO,Obrigatório,3,3,0,0,202417853018
6,2024-09-13,15:41:03,2024,RR,3018,BOA VISTA,5,702,2046,ESCOLA MUNICIPAL RAIMUNDO ELOY GOMES,...,2,NÃO,2,NÃO,Obrigatório,1,1,0,0,202457023018
7,2024-09-13,15:41:03,2024,RR,3107,UIRAMUTÃ,7,49,1023,ESCOLA ESTADUAL INDÍGENA SÃO SEBASTIÃO DO CAILÃ,...,-1,NÃO INFORMADO,-1,NÃO INFORMADO,Obrigatório,2,2,0,0,20247493107
8,2024-09-13,15:41:03,2024,RR,3018,BOA VISTA,1,77,1503,C.A.P.-D.V. CENTRO DE APOIO AS PESSOAS COM DEF...,...,2,NÃO,2,NÃO,Obrigatório,1,1,0,0,20241773018
9,2024-09-13,15:41:03,2024,RR,3018,BOA VISTA,5,190,1643,ESCOLA ESTADUAL MARIA SONIA DE BRITO OLIVA,...,-1,NÃO INFORMADO,-1,NÃO INFORMADO,Obrigatório,3,3,0,0,202451903018
